# T10 — Baseline clássico de contagem de ovos (OpenCV)

**Projeto 6 — GEVACZ | Aprendizado de Máquina**

Contagem de possíveis ovos **sem IA**, só com processamento clássico de imagens. Serve de referência (MAE) para comparar com o YOLO (ADR-005).

> Este notebook não contém fotos. Envie as imagens na seção 2. As contagens manuais ficam na seção 6.
> **Execute sempre do início ao fim, na ordem.**

## 1. Importar bibliotecas

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from IPython.display import display

print("OpenCV:", cv2.__version__)
print("Pandas:", pd.__version__)

## 2. Enviar as imagens de teste

Selecione `.jpg`, `.jpeg`, `.png` ou um `.zip`. Elas vão para `fotos/`.
Não suba fotos ao Git. **Origem das imagens:** _(preencher: quem fotografou, equipamento, data)_.

In [ ]:
from google.colab import files
import zipfile
import io

PASTA_FOTOS = Path("fotos")
PASTA_FOTOS.mkdir(exist_ok=True)

enviados = files.upload()

for nome, conteudo in enviados.items():
    if nome.lower().endswith(".zip"):
        with zipfile.ZipFile(io.BytesIO(conteudo)) as zip_ref:
            zip_ref.extractall(PASTA_FOTOS)
    elif Path(nome).suffix.lower() in [".jpg", ".jpeg", ".png"]:
        with open(PASTA_FOTOS / Path(nome).name, "wb") as f:
            f.write(conteudo)

fotos = sorted(
    p for p in PASTA_FOTOS.rglob("*")
    if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
)
print(f"Imagens encontradas: {len(fotos)}")
for foto in fotos:
    print("-", foto)

## 3. Função de contagem clássica

Etapas:
1. Achar o **disco de papel** (Otsu + maior componente conexo) e ignorar a moldura escura da lente. Sem isso o Otsu global falhou (máscara quase vazia, 103).
2. Usar o **canal vermelho**: ovos azul-escuros ficam escuros nele; o papel amarelado fica claro.
3. Desfoque gaussiano 5×5.
4. **Limiar adaptativo** gaussiano invertido (corte local, por vizinhança), porque a iluminação é desigual.
5. Abertura morfológica 3×3 para remover pontinhos.
6. Componentes conexos; conta os com área entre `area_min` e `area_max`.

**Limitações:** estrias do papel podem virar "ovo"; ovos encostados formam um só componente; não há score de confiança.

In [ ]:
def mascara_do_circulo(cinza):
    """Devolve uma máscara (255 dentro do disco de papel, 0 fora)."""
    suave = cv2.GaussianBlur(cinza, (21, 21), 0)
    _, claro = cv2.threshold(suave, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Maior ilha branca = disco de papel
    n, rotulos, stats, _ = cv2.connectedComponentsWithStats(claro, connectivity=8)
    maior = 1 + np.argmax(stats[1:, cv2.CC_STAT_AREA])
    disco = (rotulos == maior).astype(np.uint8) * 255

    # Preenche buracos (os ovos escuros são "furos" no disco)
    contornos, _ = cv2.findContours(disco, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cheio = np.zeros_like(disco)
    cv2.drawContours(cheio, contornos, -1, 255, thickness=cv2.FILLED)

    # Encolhe para não pegar a borda escura da lente
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (25, 25))
    return cv2.erode(cheio, kernel)


def contar_ovos(caminho, area_min=30, area_max=600, block_size=101, C=20):
    """Devolve (contagem, máscara binária, áreas de todas as ilhas)."""
    img = cv2.imread(str(caminho))
    if img is None:
        raise ValueError(f"Não foi possível abrir a imagem: {caminho}")

    cinza = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    circulo = mascara_do_circulo(cinza)
    canal_r = img[:, :, 2]          # OpenCV usa B, G, R -> índice 2 é o vermelho

    suave = cv2.GaussianBlur(canal_r, (5, 5), 0)
    binaria = cv2.adaptiveThreshold(
        suave, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
        cv2.THRESH_BINARY_INV, block_size, C
    )

    binaria = cv2.bitwise_and(binaria, binaria, mask=circulo)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
    binaria = cv2.morphologyEx(binaria, cv2.MORPH_OPEN, kernel)

    n, rotulos, stats, _ = cv2.connectedComponentsWithStats(binaria, connectivity=8)
    areas = stats[1:, cv2.CC_STAT_AREA]
    validas = (areas >= area_min) & (areas <= area_max)
    return int(validas.sum()), binaria, areas

## 4. Parâmetros finais

Calibrados em **uma única foto** (`ovitrampa2.jpg`), então o erro nela é otimista. Justificativa na seção 7.

In [ ]:
PARAMS = dict(
    area_min=30,      # ilhas menores (px) = ruído
    area_max=600,     # ilhas maiores (px) = ovos fundidos ou sujeira
    block_size=101,   # janela do limiar adaptativo (ímpar)
    C=20,             # quão mais escuro que a vizinhança o pixel precisa ser
)
print(PARAMS)

### 4.1 Calibração (exploratória)

Tudo abaixo usa `PARAMS` e só troca o que está sendo testado. Gabarito manual: ~390.

In [ ]:
foto_calib = fotos[0]

# Histograma das áreas das ilhas (usa os parâmetros finais)
contagem, mascara, areas = contar_ovos(foto_calib, **PARAMS)
plt.hist(areas, bins=60, range=(0, 1500))
plt.axvline(PARAMS["area_min"], color="r"); plt.axvline(PARAMS["area_max"], color="r")
plt.xlabel("Área da ilha (pixels)"); plt.ylabel("Quantidade")
plt.title(f"{foto_calib.name} — linhas vermelhas: area_min / area_max")
plt.show()

In [ ]:
# Varredura block_size x C
linhas = []
for bs in [31, 51, 71, 101, 151]:
    for c in [10, 15, 20, 30]:
        n, _, _ = contar_ovos(foto_calib, **{**PARAMS, "block_size": bs, "C": c})
        linhas.append({"block_size": bs, "C": c, "contagem": n})
varredura = pd.DataFrame(linhas).pivot(index="block_size", columns="C", values="contagem")
display(varredura)
varredura.to_csv("varredura_params.csv")

In [ ]:
# Máscaras das candidatas, lado a lado
candidatas = [(101, 10), (101, 15), (101, 20)]
fig, eixos = plt.subplots(1, 3, figsize=(24, 8))
for ax, (bs, c) in zip(eixos, candidatas):
    n, m, _ = contar_ovos(foto_calib, **{**PARAMS, "block_size": bs, "C": c})
    ax.imshow(m, cmap="gray"); ax.set_title(f"bs={bs}, C={c} → {n}"); ax.axis("off")
plt.show()

## 5. Rodar o baseline em todas as imagens

Gera `contagem_baseline.csv` e salva as máscaras em `mascaras/`.

In [ ]:
PASTA_MASCARAS = Path("mascaras")
PASTA_MASCARAS.mkdir(exist_ok=True)

resultados = []
for foto in fotos:
    contagem, mascara, areas = contar_ovos(foto, **PARAMS)
    cv2.imwrite(str(PASTA_MASCARAS / f"mascara_{foto.stem}.png"), mascara)
    resultados.append({"arquivo": foto.name, "contagem_baseline": contagem})

df = pd.DataFrame(resultados)
display(df)
df.to_csv("contagem_baseline.csv", index=False)

In [ ]:
# Original x máscara (até 4 fotos)
for foto in fotos[:4]:
    img_rgb = cv2.cvtColor(cv2.imread(str(foto)), cv2.COLOR_BGR2RGB)
    contagem, mascara, _ = contar_ovos(foto, **PARAMS)

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(img_rgb); ax[0].set_title(f"Original: {foto.name}"); ax[0].axis("off")
    ax[1].imshow(mascara, cmap="gray"); ax[1].set_title(f"Máscara — baseline: {contagem}"); ax[1].axis("off")
    plt.tight_layout(); plt.show()

## 6. Contagem manual e MAE

Preencha o dicionário com as contagens manuais (uma linha por foto). Sem upload de CSV.
Anote quem contou e se a contagem é aproximada.

In [ ]:
# arquivo -> contagem manual (ADICIONE as outras fotos aqui)
CONTAGEM_MANUAL = {
    "ovitrampa2.jpg": 390,   # aproximada, 1 pessoa, ovos sobrepostos
}

manual = pd.DataFrame(
    {"arquivo": list(CONTAGEM_MANUAL), "contagem_manual": list(CONTAGEM_MANUAL.values())}
)
tabela = df.merge(manual, on="arquivo", how="inner", validate="one_to_one")
tabela["erro"] = tabela["contagem_baseline"] - tabela["contagem_manual"]
tabela["erro_absoluto"] = tabela["erro"].abs()
display(tabela)
print(f"MAE do baseline: {tabela['erro_absoluto'].mean():.2f} ovos/imagem (n={len(tabela)})")
if len(tabela) < 3:
    print("ATENÇÃO: n < 3. O MAE é anedótico; conte mais fotos e NÃO recalibre nelas.")
print("Fotos sem contagem manual:", sorted(set(df["arquivo"]) - set(manual["arquivo"])))
tabela.to_csv("comparacao_baseline_mae.csv", index=False)

## 7. Justificativa dos parâmetros e limitações observadas

**Histórico de calibração (ovitrampa2.jpg, gabarito ~390):**
- Otsu global em cinza: 103. A moldura escura da lente dominou o histograma e a máscara ficou quase vazia.
- Limiar adaptativo, `block_size=31`, `C=12`: ~1800 (cinza) / 1203 (canal vermelho). Textura do papel contada como ovo.
- `block_size=31`, `C=20→50`: 572 → 172 → 11 → 0. Nenhum `C` intermediário confiável.
- `block_size` maior melhorou: `(101,10)=358`, `(101,15)=428`, `(101,20)=423`.
- Escolhido `block_size=101, C=20`. **Diferença entre `C=15` e `C=20` é irrelevante** (428 vs 423); escolha sem diferença técnica forte.

**Limitações vistas nas máscaras** _(conferir e ajustar com o que você vê)_:
- Estrias diagonais do papel aparecem como regiões brancas (falsos positivos).
- Ovos encostados formam uma região só (subcontagem ou descarte por `area_max`).
- O 423 pode ser coincidência de erros que se compensam.
- Calibrado em **1 foto**: o erro é otimista; não generaliza.
- Gabarito manual aproximado (1 pessoa).
- Sem score de confiança, então não alimenta o limiar de aceitação automática.

**Conclusão:** o baseline não separa ovos de textura do papel de forma confiável; serve de referência de MAE para o YOLO.

## 8. Checklist antes de entregar

- [ ] Notebook roda do início ao fim, na ordem (Executar tudo)
- [ ] Origem das imagens preenchida (seção 2)
- [ ] `contagem_baseline.csv` gerado com `PARAMS` finais
- [ ] 3 a 4 comparações original × máscara examinadas
- [ ] Erros documentados (seção 7)
- [ ] Contagens manuais de pelo menos 3 fotos (seção 6)
- [ ] MAE calculado em fotos **não usadas na calibração**
- [ ] Saídas e metadados limpos (`nbstripout`)
- [ ] Fotos fora do Git
- [ ] Branch `ml/baseline-classico` e PR com 1 aprovação